In [0]:
# jdnhs_nss_themes_org  
 
from pyspark.sql.functions import (col,trim,when,to_date,lit,current_timestamp,lower,upper,round)  
  
bronze_jdnhs_nss_themes_org_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("abfss://data@jdnhsbronze.dfs.core.windows.net/dbo.jdnhs_nss_themes_org.csv")
)

bronze_jdnhs_nss_themes_org_df.show(10)

bronze_jdnhs_nss_themes_org_df.printSchema()


In [0]:
#jdnhs_nss_people_promise_org.csv
# jdnhs_nss_themes_org.csv

# rename columns
bronze_jdnhs_nss_themes_org_df = (
    bronze_jdnhs_nss_themes_org_df
    .withColumnRenamed("base_n", "total_responses")
)

# standardise data
silver_jdnhs_nss_themes_org_df = (
    bronze_jdnhs_nss_themes_org_df
    .withColumn("ods_code", upper(trim(col("ods_code"))))
    .withColumn("org_name", lower(trim(col("org_name"))))
    .withColumn("benchmarking_group", lower(trim(col("benchmarking_group"))))
    .withColumn("region", lower(trim(col("region"))))
    .withColumn("ics", lower(trim(col("ics"))))
    .withColumn("weighting_method", lower(trim(col("weighting_method"))))
    .withColumn("metric_code", upper(trim(col("metric_code"))))
    .withColumn("metric_label", lower(trim(col("metric_label"))))
    .withColumn("score", trim(col("score")))
    .withColumn("total_responses", trim(col("total_responses")))
    .withColumn("source_file", lower(trim(col("source_file"))))
    .withColumn("source_sheet", lower(trim(col("source_sheet"))))
)

# tidy data
silver_jdnhs_nss_themes_org_df = (
    silver_jdnhs_nss_themes_org_df

    .withColumn(
        "score",
        when(col("score") == ".", None)
        .otherwise(col("score"))
        .cast("double")
    )

    .withColumn(
        "score",
        round(col("score"), 2)
    )

    .withColumn(
        "region",
        when(col("region") == "-", None)
        .otherwise(col("region"))
    )

    .withColumn(
        "ics",
        when(col("ics") == "-", None)
        .otherwise(col("ics"))
    )

    .withColumn(
        "total_responses",
        when(col("total_responses") == ".", None)
        .otherwise(col("total_responses"))
        .cast("integer")
    )
)

# check data
silver_jdnhs_nss_themes_org_df.show(30)






In [0]:
silver_jdnhs_nss_themes_org_df.select("org_name").distinct().show()

In [0]:

# validate data
valid_jdnhs_nss_themes_org_df = (
    silver_jdnhs_nss_themes_org_df.filter(
        col("ods_code").isNotNull() &
        col("org_name").isNotNull() &
        col("metric_code").isNotNull() &
        col("metric_label").isNotNull() &
        col("region").isNotNull() &
        (
            col("score").isNull() |
            col("score").between(0, 10)
        ) &
        (
            col("total_responses").isNull() |
            (col("total_responses") >= 0)
        ) &
        col("source_file").isNotNull() &
        col("source_sheet").isNotNull() &
        col("load_timestamp").isNotNull()
    )
)

quarantine_jdnhs_nss_themes_org_df = (
    silver_jdnhs_nss_themes_org_df.filter(
        col("ods_code").isNull() |
        col("org_name").isNull() |
        col("metric_code").isNull() |
        col("metric_label").isNull() |
        col("region").isNull() |
        (
            col("score").isNotNull() &
            ~col("score").between(0, 10)
        ) |
        (
            col("total_responses").isNotNull() &
            (col("total_responses") < 0)
        ) |
        col("source_file").isNull() |
        col("source_sheet").isNull() |
        col("load_timestamp").isNull()
    )
)

print(
    "Number of Quarantine Rows: ",
    quarantine_jdnhs_nss_themes_org_df.count()
)

print(
    "Number of Valid Rows: ",
    valid_jdnhs_nss_themes_org_df.count()
)



In [0]:

# see actual quarantine and valid tables

valid_jdnhs_nss_themes_org_df.show(10)

quarantine_jdnhs_nss_themes_org_df.show(10)




In [0]:

# write data 

valid_jdnhs_nss_themes_org_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "path",
        "abfss://silver-tables@jdnhsbronze.dfs.core.windows.net/themes_nss/"
    ) \
    .saveAsTable(
        "nss_themes"
    )

# Invalid data → Quarantine
(
    quarantine_jdnhs_nss_themes_org_df.write
    .format("delta")
    .mode("overwrite")
    .save(
        "abfss://quarantine@jdnhsbronze.dfs.core.windows.net/themes_nss/"
    )
)







In [0]:
from datetime import datetime

source_container = "data"
archive_container = "archive"

file_name = "dbo.jdnhs_nss_people_promise_org.csv"

source_path = f"abfss://{source_container}@jdnhsbronze.dfs.core.windows.net/{file_name}"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archived_file_name = file_name.replace(".csv", f"_{timestamp}.csv")
archive_path = f"abfss://{archive_container}@jdnhsbronze.dfs.core.windows.net/{archived_file_name}"

dbutils.fs.mv(source_path, archive_path)

print(f"Archived: {source_path} -> {archive_path}")